# 🥈 Implementando a Camada Silver

In [0]:
%python
# Imports
import pyspark.sql.functions as F
import pyspark.pandas as ps
import pandas as pd
import warnings

In [0]:
%python
# Inicializando a sessão do Spark
spark = SparkSession.builder.getOrCreate()

In [0]:
# Silencia os avisos de índice da API Pandas on Spark
warnings.filterwarnings("ignore", category=ps.utils.PandasAPIOnSparkAdviceWarning)

---

### 🤔⚒ Lógica de Tratamento (Silver)

### ⚒ Carga e Tratamento: Clientes e Produtos

In [0]:
%python
# CLIENTES
pdf_clientes = spark.read.table("bronze_clientes").pandas_api()
pdf_clientes['nome'] = pdf_clientes['nome'].str.title().str.strip()
pdf_clientes['estado'] = pdf_clientes['estado'].str.upper()
pdf_clientes = pdf_clientes[pdf_clientes['estado'] != 'XX']
pdf_clientes['email'] = pdf_clientes['email'].fillna('nao_informado@papelaria.com')

# PRODUTOS
pdf_produtos = spark.read.table("bronze_produtos").pandas_api()
pdf_produtos = pdf_produtos[pdf_produtos['preco'] > 0]
pdf_produtos = pdf_produtos[pdf_produtos['estoque'] >= 0]

# SALVANDO sem Warnings
pdf_clientes.to_spark().write.format("delta").mode("overwrite").saveAsTable("silver_clientes")
pdf_produtos.to_spark().write.format("delta").mode("overwrite").saveAsTable("silver_produtos")

print("✨ Tabelas de Clientes e Produtos tratadas com sucesso!")

✨ Tabelas de Clientes e Produtos tratadas com sucesso!


### ⚒ Carga e Tratamento: Pedidos e Itens

In [0]:
%python
# PEDIDOS
pdf_pedidos = spark.read.table("bronze_pedidos").pandas_api()

# Timestamp -> Usando o pandas padrão (pd)
pdf_pedidos['data_pedido'] = ps.to_datetime(pdf_pedidos['data_pedido'])
data_agora = pd.Timestamp.now() 
pdf_pedidos = pdf_pedidos[pdf_pedidos['data_pedido'] <= data_agora]

pdf_pedidos['status'] = pdf_pedidos['status'].fillna('Indefinido')
pdf_pedidos = pdf_pedidos[pdf_pedidos['total'] >= 0]

# ITENS PEDIDO
pdf_itens = spark.read.table("bronze_itens_pedido").pandas_api()
pdf_itens = pdf_itens[pdf_itens['quantidade'] > 0]
pdf_itens = pdf_itens[pdf_itens['preco_unit'] > 0]
pdf_itens['subtotal'] = pdf_itens['quantidade'] * pdf_itens['preco_unit']

# SALVANDO
pdf_pedidos.to_spark().write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_pedidos")

pdf_itens.to_spark().write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_itens_pedido")

print("📦 Tabelas de Pedidos e Itens tratadas com Sucesso!")

📦 Tabelas de Pedidos e Itens tratadas com Sucesso!
